<a href="https://colab.research.google.com/github/Shoaib5553/mirrordb/blob/main/mirrordb_privacy_first_synthetic_data_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Python dependencies
!pip install -q ctgan streamlit scikit-learn seaborn matplotlib pandas numpy

# Install Localtunnel via npm for free deployment
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.5 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from ctgan import CTGAN
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import torch
import io

# --- 1. UI & Setup ---
st.set_page_config(page_title="TwinTable", layout="wide")
st.title("🧬 TwinTable: Privacy-First Synthetic Data Generator")
st.markdown("Generate statistically identical, privacy-preserving tabular data using CTGAN on a T4 GPU.")

@st.cache_data
def load_data():
    """Loads the Adult Census Income dataset."""
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
               'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
               'hours-per-week', 'native-country', 'income']
    df = pd.read_csv(url, names=columns, na_values=" ?", skipinitialspace=True)
    df.dropna(inplace=True)
    return df.sample(5000, random_state=42) # Sampled down for quick Free-Tier training

# --- 2. The GAN Engine ---
class CTGANProcessor:
    def __init__(self, epochs=50):
        self.epochs = epochs
        # Explicitly check and use T4 GPU if available
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = CTGAN(epochs=self.epochs, verbose=True)

    def train_and_generate(self, data, discrete_columns):
        with st.spinner(f"Training CTGAN on {self.device.upper()} (this takes a few minutes)..."):
            self.model.fit(data, discrete_columns)
        with st.spinner("Generating Synthetic Twin..."):
            return self.model.sample(len(data))

# --- 3. The Validation Suite ---
def plot_distributions(real_df, syn_df, column):
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.kdeplot(real_df[column], label='Real', fill=True, ax=ax)
    sns.kdeplot(syn_df[column], label='Synthetic', fill=True, ax=ax)
    ax.set_title(f"Distribution Comparison: {column}")
    ax.legend()
    return fig

def plot_correlation_heatmaps(real_df, syn_df):
    # Encode categorical columns for correlation math
    real_enc, syn_enc = real_df.copy(), syn_df.copy()
    for col in real_enc.columns:
        if real_enc[col].dtype == 'object':
            le = LabelEncoder()
            real_enc[col] = le.fit_transform(real_enc[col].astype(str))
            syn_enc[col] = le.transform(syn_enc[col].astype(str))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(real_enc.corr(), cmap="coolwarm", ax=axes[0], cbar=False)
    axes[0].set_title("Real Data Correlation")
    sns.heatmap(syn_enc.corr(), cmap="coolwarm", ax=axes[1], cbar=False)
    axes[1].set_title("Synthetic Data Correlation")
    return fig

def calculate_utility_score(real_df, syn_df, target_col='income'):
    # Prepare data for ML Utility test
    df_real, df_syn = real_df.copy(), syn_df.copy()

    # Label Encoding
    for col in df_real.columns:
        if df_real[col].dtype == 'object':
            le = LabelEncoder()
            df_real[col] = le.fit_transform(df_real[col].astype(str))
            df_syn[col] = df_syn[col].map(lambda s: '<unknown>' if s not in le.classes_ else s)
            le.classes_ = np.append(le.classes_, '<unknown>')
            df_syn[col] = le.transform(df_syn[col].astype(str))

    # Real Model
    X_real = df_real.drop(columns=[target_col])
    y_real = df_real[target_col]
    Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_real, y_real, test_size=0.2, random_state=42)

    clf_real = RandomForestClassifier(random_state=42)
    clf_real.fit(Xr_train, yr_train)
    real_acc = accuracy_score(yr_test, clf_real.predict(Xr_test))

    # Synthetic Model (Train on Synthetic, Test on Real holdout)
    X_syn = df_syn.drop(columns=[target_col])
    y_syn = df_syn[target_col]
    Xs_train, _, ys_train, _ = train_test_split(X_syn, y_syn, test_size=0.2, random_state=42)

    clf_syn = RandomForestClassifier(random_state=42)
    clf_syn.fit(Xs_train, ys_train)
    syn_acc = accuracy_score(yr_test, clf_syn.predict(Xr_test))

    return real_acc, syn_acc

# --- 4. Frontend Execution ---
real_data = load_data()
discrete_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

col1, col2 = st.columns([1, 3])

with col1:
    st.subheader("Data Setup")
    st.dataframe(real_data.head(), use_container_width=True)
    epochs = st.slider("Training Epochs", min_value=10, max_value=300, value=50, step=10)
    start_training = st.button("Train TwinTable Model", type="primary")

with col2:
    if start_training:
        processor = CTGANProcessor(epochs=epochs)
        syn_data = processor.train_and_generate(real_data, discrete_cols)

        st.success("Training Complete! Synthetic Data Generated.")

        # Validation Visuals
        st.subheader("📊 Validation Suite")
        st.pyplot(plot_distributions(real_data, syn_data, 'age'))
        st.pyplot(plot_correlation_heatmaps(real_data, syn_data))

        # Utility Score
        real_acc, syn_acc = calculate_utility_score(real_data, syn_data)
        st.subheader("🤖 Machine Learning Utility Test")
        st.markdown(f"**Random Forest Accuracy (Trained on Real):** {real_acc:.2%}")
        st.markdown(f"**Random Forest Accuracy (Trained on Synthetic):** {syn_acc:.2%}")
        st.progress(syn_acc / real_acc, text=f"Utility Retention: {(syn_acc/real_acc):.2%}")

        # Download Button
        csv = syn_data.to_csv(index=False).encode('utf-8')
        st.download_button(label="Download Synthetic Dataset (CSV)", data=csv, file_name='synthetic_adult_data.csv', mime='text/csv')

Writing app.py


In [3]:
import urllib
# Extract the Colab machine's IP address (You will need to paste this into the Localtunnel security page)
colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"✅ Your Endpoint IP for Localtunnel is: {colab_ip}")
print("Click the link below, and paste the IP address when prompted to view your dashboard.")

# Run Streamlit in the background and expose port 8501 via localtunnel
!streamlit run app.py & npx localtunnel --port 8501

✅ Your Endpoint IP for Localtunnel is: 34.169.236.110
Click the link below, and paste the IP address when prompted to view your dashboard.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧your url is: https://cyan-camels-sort.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.169.236.110:8501

2026-03-03 14:54:21.059 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-03 14:54:56.117 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-03 14:55:14.213 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 202